Generate a chart for a validation song and compare it with the ground truth

Runs top to bottom: setup, load validation data, load model, generate, export.

In [ ]:
from pathlib import Path
import torch

from tracers.datasets import AutoOsu4KDataset
from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]
device = setup["device"]

In [ ]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    segment_frames=segment_frames,
    seed=42,
)

In [ ]:
from tracers.pipelines.generate import run_validation_generation

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=str(PROJECT_ROOT / "notebooks" / "checkpoints" / "best.pt"),
    train_loader=None,
    valid_loader=None,
)


Resumed from epoch 76. Best validation loss: 0.1494


In [ ]:
context_frames = 500

val_dataset = AutoOsu4KDataset(
    beatmap_dir=beatmap_dir,
    audio_dir=audio_dir,
    segment_frames=segment_frames,
    crop_mode="all",
    stride_frames=segment_frames - context_frames,
    song_ids=val_ids,
    audio_cache_size=8,
)

val_loader = build_loader_for_model(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

In [ ]:
# ---------- settings ----------
CHART_ID = None          # e.g. "1055661-148-4"; None = first chart in the validation set
BPM = 120                # only used to place notes in the exported JSON
OFFSET_MS = 0
LANE_MAP = (0, 2, 4, 6)

TEMPERATURE = None        # None = greedy
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- collect the chart's segments ----------
INPUT_KEYS = ["specs", "beat_phase", "beat_num", "difficulty", "valid_frames"]
segments, found = [], False

for batch in val_loader:
    for i, cid in enumerate(batch["chart_id"]):
        if CHART_ID is None:
            CHART_ID = cid
        if cid != CHART_ID:
            if found:
                break
            continue
        found = True
        segments.append({
            "start": int(batch["start_frame"][i]),
            "sample": {k: batch[k][i:i+1] for k in INPUT_KEYS},
            "actual": batch["actions_target"][i].cpu(),
        })
    else:
        continue
    break

segments.sort(key=lambda s: s["start"])
print(f"Chart {CHART_ID}: {len(segments)} segments")

Chart 1055661-148-4: 8 segments


In [ ]:
generated, actual, debug_segments = run_validation_generation(
    trainer,
    val_loader,
    chart_id=CHART_ID,
    context_frames=context_frames,
    seed_action=None,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    seed=SEED,
    collect_debug=SAVE_DEBUG_CSV,
)

Frame      0 | frames 2000 | active   23 (actual   68) | next seed 0
Frame   2000 | frames 2000 | active    8 (actual   56) | next seed 0
Frame   4000 | frames 2000 | active   17 (actual   50) | next seed 0
Frame   6000 | frames 2000 | active   36 (actual   50) | next seed 0
Frame   8000 | frames 2000 | active   52 (actual   64) | next seed 0
Frame  10000 | frames 2000 | active   61 (actual   63) | next seed 0
Frame  12000 | frames 2000 | active   40 (actual   57) | next seed 0
Frame  14000 | frames 946 | active    0 (actual   12) | next seed 0

Done. 14946 frames (149.5 s)
Active frames: generated 237 | actual 420


In [ ]:
from tracers.export.chart import export_generated_chart

first_start = segments[0]["start"] if segments else 0
valid_all = torch.ones(len(generated), dtype=torch.bool)

output_dir = Path(f"charts/{CHART_ID}")
output_dir.mkdir(parents=True, exist_ok=True)

export_generated_chart(
    generated,
    output_dir / "generated_np_chart.json",
    bpm=BPM,
    start_frame=first_start,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
)

export_generated_chart(
    actual,
    output_dir / "actual_full_chart.json",
    bpm=BPM,
    start_frame=first_start,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
)

print("Saved generated_np_chart.json and actual_full_chart.json")

if SAVE_DEBUG_CSV:
    from tracers.generate.converter import export_generation_debug
    export_generation_debug(debug_segments, output_path=str(output_dir / "generation_np_debug.csv"), top_k=5)

Saved generated_np_chart.json and actual_full_chart.json
Exported 14946 frame records to: charts\1055661-148-4\generation_np_debug.csv
